In [1]:
import os
from pathlib import Path

# set the root directory as the current working directory
os.chdir(Path.cwd().parent)
print(f"Current working directory: {os.getcwd()}")

Current working directory: d:\Programing\CyberSec-Reasoner


In [30]:
import re
import random
import warnings
from pprint import pprint
from datasets import load_dataset, load_from_disk, concatenate_datasets, DatasetDict

warnings.filterwarnings("ignore")

*****
# Download Huggingface Datasets and save to Raw folder

In [3]:
load_dataset("AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.0").save_to_disk("data/raw/Cybersecurity-Dataset-Fenrir-v2.0")
load_dataset("Mohannadcse/cybersec-reasoning-merged").save_to_disk("data/raw/cybersec-reasoning-merged")
load_dataset("trendmicro-ailab/Primus-Reasoning").save_to_disk("data/raw/Primus-Reasoning")

Generating train split: 83920 examples [00:01, 74280.95 examples/s]
Saving the dataset (1/1 shards): 100%|██████████| 23146/23146 [00:00<00:00, 295226.39 examples/s]
Generating train split: 4891 examples [00:00, 60873.96 examples/s]
Saving the dataset (1/1 shards): 100%|██████████| 4891/4891 [00:00<00:00, 265599.07 examples/s]


*****
# Merge Fenrir & Cybersec datasets for SFT

In [4]:
fenrir = load_from_disk("data/raw/Cybersecurity-Dataset-Fenrir-v2.0")
print(fenrir)
cybersec = load_from_disk("data/raw/cybersec-reasoning-merged")
print(cybersec)

DatasetDict({
    train: Dataset({
        features: ['system', 'user', 'assistant'],
        num_rows: 83920
    })
})
DatasetDict({
    train: Dataset({
        features: ['prompt', 'reasoning_content', 'answer'],
        num_rows: 23146
    })
})


In [17]:
pprint(fenrir["train"][random.randint(0, len(fenrir["train"]) - 1)])

{'assistant': 'Monitoring strategies for detecting operational anomalies in '
              'metamorphic malware analysis using category theory frameworks '
              'require sophisticated approaches that address the dynamic and '
              'evolving nature of these threats. Traditional signature-based '
              'detection methods are insufficient due to the polymorphic '
              'transformations that metamorphic malware undergoes, altering '
              'its code structure while maintaining '
              'functionality.\\\\n\\\\n**Behavioral Analysis**: One effective '
              'strategy involves behavioral monitoring, which focuses on the '
              'actions performed by the malware rather than its static '
              "characteristics. This approach aligns with category theory's "
              'emphasis on morphisms and transformations, where the focus is '
              'on how objects (malware instances) interact within a category '
          

In [21]:
pprint(cybersec["train"][random.randint(0, len(cybersec["train"]) - 1)])

{'answer': "If you don't address this issue, your system could be at risk of "
           "being compromised by external changes to the tools you're relying "
           'on. This might allow unauthorized individuals to alter how your '
           'processes work, potentially leading to security breaches, data '
           'exposure, or unintended actions that could harm your operations.',
 'prompt': 'cwe_id:CWE-829\n'
           'cwe_name:Inclusion of Functionality from Untrusted Control Sphere\n'
           'affected_line:Unpinned Actions Full Length Commit SHA\n'
           'partial_code:- name: Docker meta\n'
           '        id: meta\n'
           '        uses: docker/metadata-action@v3\n'
           '        with:\n'
           '          images: |\n'
           '            lablabs/cloudflare_exporter\n'
           '            ghcr.io/lablabs/cloudflare_exporter\n'
           '          # generate Docker tags based on the following '
           'events/attributes\n'
       

In [5]:
def format_fenrir(data):
    messages = [
        {"role": "system", "content": data["system"]},
        {"role": "user", "content": data["user"]},
        {"role": "assistant", "content": data["assistant"]},
    ]
    return {"messages": messages}

In [6]:
def format_cybersec_dataset(data, system_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": data['prompt']},
        {"role": "assistant", "content": f"<think>\n{data['reasoning_content']}\n</think>\n\n{data['answer']}"},
    ]
    return {"messages": messages}

In [7]:
fenrir = fenrir.map(
    format_fenrir,
    remove_columns=fenrir["train"].column_names
)

system_prompt = ("You are a cybersecurity expert. Analyze the given vulnerability context carefully and "
                "think step by step to understand the root cause, risk, and impact. Then provide a clear, "
                "concise, and accurate answer to the user's question based on your reasoning.")

cybersec = cybersec.map(
    lambda x: format_cybersec_dataset(x, system_prompt),
    remove_columns=cybersec["train"].column_names
)

Map: 100%|██████████| 23146/23146 [00:01<00:00, 17462.91 examples/s]


In [8]:
pprint(cybersec["train"][1545])

{'messages': [{'content': 'You are a cybersecurity expert. Analyze the given '
                          'vulnerability context carefully and think step by '
                          'step to understand the root cause, risk, and '
                          'impact. Then provide a clear, concise, and accurate '
                          "answer to the user's question based on your "
                          'reasoning.',
               'role': 'system'},
              {'content': 'cwe_id:CWE-200\n'
                          'cwe_name:Exposure of Sensitive Information to an '
                          'Unauthorized Actor\n'
                          'affected_line:Exposure of Sensitive Information to '
                          'an Unauthorized Actor in SonarSource SonarQube API\n'
                          'partial_code:org.sonarsource.sonarqube:sonar-plugin-api '
                          '6.3\n'
                          'file_name:pom.xml\n'
                          'status:True P

In [9]:
dataset = concatenate_datasets([fenrir["train"], cybersec["train"]])
dataset

Dataset({
    features: ['messages'],
    num_rows: 107066
})

In [10]:
dataset = dataset.train_test_split(test_size=0.10, seed=42, shuffle=True)
split = dataset["test"].train_test_split(test_size=0.50, seed=42, shuffle=True)
dataset = DatasetDict({
    "train": dataset["train"],
    "validation": split["train"],
    "test": split["test"]
})

dataset.save_to_disk("data/processed/cybersecurity_sft_dataset")
print(dataset)

Saving the dataset (1/1 shards): 100%|██████████| 5354/5354 [00:00<00:00, 150749.87 examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 96359
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 5353
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 5354
    })
})


*****
# Prepare Primus Dataset for R-SFT & GRPO

In [50]:
primus = load_from_disk("data/raw/Primus-Reasoning")
primus

DatasetDict({
    train: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 4891
    })
})

In [52]:
primus["train"][random.randint(0, len(primus["train"]) - 1)]["messages"]

[{'content': 'Analyze the following CVE description and map it to the appropriate CWE. Provide a brief justification for your choice. Ensure the last line of your response contains only the CWE ID.  CVE Description: Adobe Acrobat and Reader versions 2019.021.20061 and earlier, 2017.011.30156 and earlier, 2017.011.30156 and earlier, and 2015.006.30508 and earlier have an use after free vulnerability. Successful exploitation could lead to arbitrary code execution . ',
  'role': 'user'},
 {'content': "<|reserved_special_token_0|>**Initial Analysis of CVE Description**\nUnderstanding the CVE description provided: Adobe Acrobat and Reader versions have a 'use after free' vulnerability which can lead to arbitrary code execution.\n**Identifying Key Information**\nThe key elements are 'use after free' vulnerability and 'arbitrary code execution'. 'Use after free' is a specific type of memory corruption issue.\n**Mapping to Potential CWE IDs**\nConsidering CWEs related to 'use after free':\n- C

In [53]:
def format_primus(data, system_prompt):
    data = data["messages"]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": data[0]["content"]},
        {"role": "assistant", "content": (data[1]["content"]).replace("<|reserved_special_token_0|>", "<think>\n").replace("<|reserved_special_token_1|>", "</think>")},
    ]
    return {"messages": messages}

In [54]:
system_prompt = """You are a cybersecurity reasoning expert specialized in vulnerability analysis and classification.

Your task is to analyze CVE (Common Vulnerabilities and Exposures) descriptions and map them to the most appropriate CWE (Common Weakness Enumeration).

You must follow a structured reasoning workflow:

1. Understand the vulnerability context and affected components  
2. Identify the vulnerability type (e.g., XSS, memory corruption, improper validation)  
3. Determine the root cause of the weakness  
4. Map the root cause to the most appropriate CWE category  

Guidelines:
- Focus on root cause rather than surface-level keywords  
- Use precise cybersecurity terminology  
- Ensure reasoning clearly supports the final CWE selection  
- Prefer the most specific applicable CWE when possible  

Avoid:
- Guessing without justification  
- Contradictions between reasoning and conclusion  
- Irrelevant or overly generic explanations  
- Blindly copying CWE identifiers from the input  

Output Format (STRICT):

1. First, provide detailed step-by-step reasoning inside:
   <think> ... </think>

2. Then provide a structured analytical explanation (clear, well-organized, 1–2 paragraphs) that summarizes:
   - the vulnerability type  
   - the root cause  
   - why the selected CWE is the best match  

3. The last line must contain ONLY the CWE ID (e.g., CWE-79)

Important:
- The explanation must be consistent with the reasoning  
- The CWE must be fully justified by both reasoning and explanation  
- Do not include anything after the CWE ID  
"""

In [55]:
primus = primus.map(
    lambda x: format_primus(x, system_prompt),
    remove_columns=primus["train"].column_names
)

primus.save_to_disk("data/processed/primus_reasoning_dataset")
print(primus)

Saving the dataset (1/1 shards): 100%|██████████| 4891/4891 [00:00<00:00, 285305.77 examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 4891
    })
})


In [56]:
primus["train"][random.randint(0, len(primus["train"]) - 1)]

{'messages': [{'content': 'You are a cybersecurity reasoning expert specialized in vulnerability analysis and classification.\n\nYour task is to analyze CVE (Common Vulnerabilities and Exposures) descriptions and map them to the most appropriate CWE (Common Weakness Enumeration).\n\nYou must follow a structured reasoning workflow:\n\n1. Understand the vulnerability context and affected components  \n2. Identify the vulnerability type (e.g., XSS, memory corruption, improper validation)  \n3. Determine the root cause of the weakness  \n4. Map the root cause to the most appropriate CWE category  \n\nGuidelines:\n- Focus on root cause rather than surface-level keywords  \n- Use precise cybersecurity terminology  \n- Ensure reasoning clearly supports the final CWE selection  \n- Prefer the most specific applicable CWE when possible  \n\nAvoid:\n- Guessing without justification  \n- Contradictions between reasoning and conclusion  \n- Irrelevant or overly generic explanations  \n- Blindly co

In [ ]:
# keep only mapping data that ends with CWE-xxx and remove mcq or other data without CWE mapping
pattern = re.compile(r"\n\nCWE-\d+\s*$")

def filter_cwe_data(data):
    """ Filter out data that does not end with a CWE mapping in the assistant's response. """
    try:
        content = data["messages"][-1]["content"]
        return bool(pattern.search(content))
    except:
        return False

In [58]:
filtered_dataset = primus.filter(filter_cwe_data)
print(filtered_dataset)

Filter: 100%|██████████| 4891/4891 [00:00<00:00, 36959.78 examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 2307
    })
})


In [49]:
filtered_dataset["train"][random.randint(0, len(filtered_dataset["train"]) - 1)]

{'messages': [{'content': 'You are a cybersecurity reasoning expert specialized in vulnerability analysis and classification.\n\nYour task is to analyze CVE (Common Vulnerabilities and Exposures) descriptions and map them to the most appropriate CWE (Common Weakness Enumeration).\n\nYou must follow a structured reasoning process:\n\n1. Understand the vulnerability context and affected components\n2. Identify the core vulnerability type (e.g., XSS, SQL Injection, improper validation)\n3. Determine the root cause (not just symptoms)\n4. Map the root cause to the most appropriate CWE category\n\nGuidelines:\n- Focus on root cause, not surface-level keywords\n- Use precise cybersecurity terminology\n- Ensure reasoning directly supports the final CWE\n- Prefer the most specific applicable CWE when possible\n\nAvoid:\n- Guessing or unsupported conclusions\n- Contradictions between reasoning and final answer\n- Irrelevant or overly generic explanations\n- Relying only on explicit CWE mentions

In [59]:
filtered_dataset.save_to_disk("data/processed/primus_reasoning_cwe_mapping_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 2307/2307 [00:00<00:00, 31230.46 examples/s]
